# Step 3 — Vectorization & FAISS Index

This notebook reads `chunks/chunks.csv`, encodes every chunk into a vector using each of the
three fine-tuned encoders, and saves one FAISS index per model.

**Prerequisite:** `baseline_step1.ipynb` must have been run with weight saving enabled, producing:
```
weights/bert-base-uncased_IHC/   or   weights/bert-base-uncased_ISHate/
weights/hateBERT_IHC/            or   weights/hateBERT_ISHate/
weights/roberta-base_IHC/        or   weights/roberta-base_ISHate/
```

**Outputs:**
```
index/bert-base-uncased.faiss
index/hateBERT.faiss
index/roberta-base.faiss
index/documents.json    ← lookup table: {chunk_id → text}
```

**Key design choices:**
- Encoder: `AutoModel` (base transformer only — classification head from fine-tuning is ignored)
- Embedding: `[CLS]` token — `last_hidden_state[:, 0, :]`
- Similarity: cosine, via L2-normalization + `IndexFlatIP` (exact search)
- Index type: `IndexIDMap(IndexFlatIP)` — FAISS returns `chunk_id` directly, not a positional row index
- Embedding dimension: read from `model.config.hidden_size`, asserted at runtime

## 1. Imports

In [1]:
import os
import json
import numpy as np
import pandas as pd
import torch
import faiss
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

## 2. Configuration

Edit `DATASET_USED` to switch between fine-tuned checkpoints (IHC vs ISHate).
The HuggingFace model ID (`hf_fallback`) is used if the local weights folder does not exist yet.

> **Known limitation — encoder/index mismatch:** `chunks.csv` contains data from both IHC and ISHate,
> but each encoder is fine-tuned on only one of them. This is accepted for now given the small domain
> gap (both are Twitter hate speech). Two cleaner solutions to revisit later:
> - **Joint fine-tuning:** retrain one model on IHC + ISHate combined, build a single unified index.
> - **Separate indexing:** one index per dataset, each encoded by its matching fine-tuned model, then merge top-k results at query time.

In [2]:
DATASET_USED = "Vicomtech"   # "IHC", "ISHate", or "Vicomtech" — determines which fine-tuned weights to load

MODEL_CONFIGS = {
    "bert-base-uncased": {
        "weights_path": f"../weights/bert-base-uncased_{DATASET_USED}",
        "hf_fallback":  "bert-base-uncased",
    },
    "hateBERT": {
        "weights_path": f"../weights/hateBERT_{DATASET_USED}",
        "hf_fallback":  "GroNLP/hateBERT",
    },
    "roberta-base": {
        "weights_path": f"../weights/roberta-base_{DATASET_USED}",
        "hf_fallback":  "roberta-base",
    },
}

BATCH_SIZE = 64
MAX_LENGTH = 64    # 50 content tokens + [CLS]/[SEP] + margin

# Similarity threshold used at retrieval time.
# If result counts vary wildly across queries (sometimes 0, sometimes 10),
# the threshold is too strict — a dedicated test notebook will validate this
# and switch to top-k if needed.
SIMILARITY_THRESHOLD = 0.7

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


## 3. Load Chunks

Load `chunks/chunks.csv`. The `text` column contains the string that will be encoded.
The `chunk_id` column is the row index and will match the FAISS vector position.

In [3]:
df = pd.read_csv("chunks/chunks.csv")[["chunk_id", "text"]]
texts     = df["text"].tolist()
chunk_ids = df["chunk_id"].to_numpy(dtype="int64")
print(f"Chunks to index: {len(texts):,}")
df.head()

Chunks to index: 63,277


,chunk_id,text
0,0,[not hate] we need everyone to keep winning . ...
1,1,[not hate] : the hatred spewed by robert spenc...
2,2,[not hate] are antifa boomers ?
3,3,[not hate] #trucons aren't capable of anything...
4,4,[not hate] wow you really caught that . so ha...


## 4. Encoding Function

Takes a list of strings + a loaded model + tokenizer, returns a float32 numpy array of shape
`(N, 768)` where each row is the `[CLS]` embedding of one chunk.

Uses batched inference with `torch.no_grad()`. Moves tensors to GPU if available.

In [4]:
def encode(texts, model, tokenizer, batch_size=BATCH_SIZE, max_length=MAX_LENGTH):
    all_vecs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding"):
        batch = texts[i : i + batch_size]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_length,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        cls_vecs = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_vecs.append(cls_vecs)
    return np.vstack(all_vecs).astype("float32")

## 5. Build One FAISS Index Per Model

For each model in `MODEL_CONFIGS`:
1. Determine which path to load from: local `weights_path` if the folder exists, else `hf_fallback`
2. Load tokenizer and model (`AutoModel`, not classification)
3. Encode all chunks (show progress with tqdm)
4. L2-normalize vectors (`faiss.normalize_L2`) for cosine similarity
5. Build `IndexFlatIP(768)` and add all vectors
6. Save index to `index/{model_name}.faiss`
7. Print: total vectors added, index size on disk

In [5]:
os.makedirs("index", exist_ok=True)

for model_name, cfg in MODEL_CONFIGS.items():
    load_path = cfg["weights_path"] if os.path.isdir(cfg["weights_path"]) else cfg["hf_fallback"]
    print(f"\n=== {model_name} — loading from: {load_path} ===")

    tokenizer = AutoTokenizer.from_pretrained(load_path)
    model     = AutoModel.from_pretrained(load_path)
    model.eval().to(device)

    vectors = encode(texts, model, tokenizer)

    embed_dim = model.config.hidden_size
    assert vectors.shape == (len(texts), embed_dim), \
        f"Unexpected shape {vectors.shape}, expected ({len(texts)}, {embed_dim})"
    print(f"  Vectors shape: {vectors.shape}  ✓")

    faiss.normalize_L2(vectors)  # unit-norm → dot product = cosine similarity

    # IndexIDMap stores chunk_ids so search() returns chunk_ids directly, not row positions
    inner_index = faiss.IndexFlatIP(embed_dim)
    index = faiss.IndexIDMap(inner_index)
    index.add_with_ids(vectors, chunk_ids)

    out_path = f"index/{model_name}_{DATASET_USED}.faiss"
    faiss.write_index(index, out_path)
    print(f"  Saved {out_path}  ({index.ntotal} vectors)")

    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()


=== bert-base-uncased — loading from: ../weights/bert-base-uncased_Vicomtech ===


Encoding:   0%|          | 0/989 [00:00<?, ?it/s]

Encoding:   0%|          | 1/989 [00:00<03:08,  5.24it/s]

Encoding:   0%|          | 3/989 [00:00<01:28, 11.19it/s]

Encoding:   1%|          | 6/989 [00:00<01:03, 15.56it/s]

...

Encoding: 100%|█████████▉| 986/989 [00:39<00:00, 24.74it/s]

Encoding: 100%|██████████| 989/989 [00:39<00:00, 25.27it/s]

Encoding: 100%|██████████| 989/989 [00:39<00:00, 25.22it/s]

  Vectors shape: (63277, 768)  ✓


  Saved index/roberta-base_Vicomtech.faiss  (63277 vectors)


## 6. Save Shared Lookup Table

FAISS returns integer positions, not text. We save `documents.json` as a list of strings
so that position `i` in the FAISS index maps to `documents[i]` in the JSON.

This lookup table is **shared across all three indexes** — the chunk order is identical.

In [6]:
# Keyed by str(chunk_id) — JSON keys are always strings
documents = {str(cid): text for cid, text in zip(chunk_ids, texts)}

with open(f"index/documents_{DATASET_USED}.json", "w") as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)

print(f"Saved index/documents_{DATASET_USED}.json  ({len(documents):,} entries)")

Saved index/documents_Vicomtech.json  (63,277 entries)


## 7. Sanity Check

Load the hateBERT index and run two sample queries to verify the pipeline end-to-end.

> If result counts vary wildly across queries (e.g. sometimes 0, sometimes 10), `SIMILARITY_THRESHOLD` is too strict. A dedicated test notebook will measure this and switch to top-k if needed.

In [7]:
test_index = faiss.read_index(f"index/hateBERT_{DATASET_USED}.faiss")
with open(f"index/documents_{DATASET_USED}.json") as f:
    documents = json.load(f)

_sc_path = (MODEL_CONFIGS["hateBERT"]["weights_path"]
            if os.path.isdir(MODEL_CONFIGS["hateBERT"]["weights_path"])
            else MODEL_CONFIGS["hateBERT"]["hf_fallback"])
tokenizer_sc = AutoTokenizer.from_pretrained(_sc_path)
model_sc     = AutoModel.from_pretrained(_sc_path).eval().to(device)

def query_index(tweet_text, k=5, threshold=SIMILARITY_THRESHOLD):
    vec = encode([tweet_text], model_sc, tokenizer_sc, batch_size=1)
    faiss.normalize_L2(vec)
    scores, ids = test_index.search(vec, k)
    return [
        (documents[str(ids[0][i])], float(scores[0][i]))
        for i in range(k)
        if ids[0][i] != -1 and scores[0][i] >= threshold
    ]

for tweet in [
    "I hate immigrants, they should all go back",
    "Let's celebrate diversity in our community",
]:
    print(f"\nQuery: {tweet}")
    hits = query_index(tweet)
    if hits:
        for text, score in hits:
            print(f"  {score:.3f}  {text}")
    else:
        print("  (no results above threshold)")


Query: I hate immigrants, they should all go back


Encoding:   0%|          | 0/1 [00:00<?, ?it/s]

Encoding: 100%|██████████| 1/1 [00:00<00:00, 103.60it/s]

  0.897  [hate] Immigrants are bad and need to be sent back to their own country.
  0.881  [hate] why should we change the way we live in a country that my ancestors break down for just for these immigrants to number in and demand change
  0.878  [hate] Immigrants are a burden to the UK and it is high time they left!
  0.877  [hate] I hate immigrants they are just a bunch of criminals.
  0.876  [hate] why should we change the way why we live in a country that my ancestors died for wherefore just for these immigrants to come in and demand change

Query: Let's celebrate diversity in our community


Encoding:   0%|          | 0/1 [00:00<?, ?it/s]

Encoding: 100%|██████████| 1/1 [00:00<00:00, 158.68it/s]

  0.748  [not hate] We have to learn to appreciate diversity .
  0.738  [not hate] it's simply more important to continue and to be seen than anything else . our message is what will win . no matter the optics ! how bout we just unite ourselves for a change ?
  0.735  [hate] we all need to doubtlessly stick together for the white race
  0.731  [hate] we all need to surely stick together for the white race
  0.731  [not hate] This is why we are here to share and grow in our craft of survival .
